# 03 — Comparación de modelos de Machine Learning

## Modelo predictivo para anticipar incumplimientos de SLA en HR Operations  
## (Predictive Model to Anticipate SLA Breaches in HR Operations)

El objetivo de este notebook es entrenar y comparar varios modelos de clasificación supervisada para predecir la variable objetivo `incumplio_sla`.

La selección del modelo final no se basará únicamente en accuracy, sino especialmente en la capacidad para detectar tickets que realmente incumplen SLA.

La métrica principal será el `recall` de la clase `1`, porque esta clase representa los casos de incumplimiento.

## Modelos a comparar

Se compararán los siguientes modelos:

| Modelo | Justificación |
|---|---|
| KNN | Modelo inicial basado en similitud entre tickets. |
| Logistic Regression | Baseline interpretable para estimar probabilidad de incumplimiento. |
| Decision Tree | Modelo explicable basado en reglas de decisión. |
| Random Forest | Modelo robusto que combina múltiples árboles para mejorar estabilidad. |
| Gradient Boosting | Modelo avanzado para capturar patrones complejos y optimizar rendimiento. |

In [1]:
# Importamos librerías
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
# Cargamos el dataset final listo para modelado:
ruta_datos = "../data/processed/tickets_hr_sla_model_ready.csv"

df = pd.read_csv(ruta_datos)

df.head()

,Customer Age,Product Purchased,Ticket Type,Ticket Subject,Ticket Priority,Ticket Channel,incumplio_sla
0,48,Dell XPS,Technical issue,Network problem,Low,Social media,0
1,27,Microsoft Office,Billing inquiry,Account access,Low,Social media,0
2,67,Autodesk AutoCAD,Billing inquiry,Data loss,Low,Email,0
3,48,Nintendo Switch,Cancellation request,Data loss,High,Phone,0
4,51,Microsoft Xbox Controller,Product inquiry,Software bug,High,Chat,1


In [3]:
# Ver tamaño:
df.shape

(2769, 7)

In [4]:
# Ver columnas:
df.columns

Index(['Customer Age', 'Product Purchased', 'Ticket Type', 'Ticket Subject',
       'Ticket Priority', 'Ticket Channel', 'incumplio_sla'],
      dtype='object')

## Selección de variables

El dataset final ya contiene únicamente variables seguras para modelado.

La variable objetivo es:

- `incumplio_sla = 0`: el ticket cumplió el SLA.
- `incumplio_sla = 1`: el ticket incumplió el SLA.

Las variables predictoras son aquellas disponibles antes del cierre del ticket.

In [5]:
X = df.drop(columns="incumplio_sla")
y = df["incumplio_sla"]

X.head()

,Customer Age,Product Purchased,Ticket Type,Ticket Subject,Ticket Priority,Ticket Channel
0,48,Dell XPS,Technical issue,Network problem,Low,Social media
1,27,Microsoft Office,Billing inquiry,Account access,Low,Social media
2,67,Autodesk AutoCAD,Billing inquiry,Data loss,Low,Email
3,48,Nintendo Switch,Cancellation request,Data loss,High,Phone
4,51,Microsoft Xbox Controller,Product inquiry,Software bug,High,Chat


In [6]:
y.value_counts(normalize=True).mul(100).round(2)

incumplio_sla
0    61.79
1    38.21
Name: proportion, dtype: float64

## Separación train/test

El dataset se divide en dos partes:

- `train`: utilizado para entrenar los modelos.
- `test`: utilizado para evaluar los modelos con datos no vistos.

Se utiliza una división 80/20 y se aplica `stratify=y` para mantener la proporción original de clases.

In [7]:
# Separamos en train y test:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")


X_train: (2215, 6)
X_test: (554, 6)
y_train: (2215,)
y_test: (554,)


## Preprocesamiento común

Para comparar los modelos de forma justa, se utiliza el mismo preprocesamiento:

- `StandardScaler` para variables numéricas.
- `OneHotEncoder` para variables categóricas.

Aunque algunos modelos basados en árboles no necesitan escalado, mantener una estructura común permite trabajar con pipelines homogéneos y reproducibles.

In [8]:
columnas_numericas = ["Customer Age"]

columnas_categoricas = [
    "Product Purchased",
    "Ticket Type",
    "Ticket Subject",
    "Ticket Priority",
    "Ticket Channel"
]

preprocesamiento = ColumnTransformer(
    transformers=[
        ("numericas", StandardScaler(), columnas_numericas),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas)
    ]
)

## Definición de modelos

Se entrenarán cinco modelos de clasificación supervisada.

Todos los modelos serán evaluados con las mismas métricas:

- Accuracy.
- Precision clase 1.
- Recall clase 1.
- F1-score clase 1.

La clase `1` representa tickets que incumplieron SLA.

In [10]:
# Modelos a comparar:
modelos = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Regresión Logística": LogisticRegression(random_state=42, max_iter=1000),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [11]:
# Entrenamos todos los modelos y guardamos sus métricas en una tabla.
resultados = []

for nombre_modelo, modelo in modelos.items():
    # Creamos pipeline
    pipeline = Pipeline(steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", modelo)
    ]
)

# Entrenamos el modelo:
    pipeline.fit(X_train, y_train)

# Hacemos predicciones:
    y_pred = pipeline.predict(X_test)

# Calculamos métricas principales:
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
# Guardamos resultados:
    resultados.append({
        "modelo": nombre_modelo,
        "accuracy": accuracy,
        "precision_clase_1": precision, 
        "recall_clase_1": recall,
        "f1_clase_1": f1
    })

resultados_modelos = pd.DataFrame(resultados)
resultados_modelos


,modelo,accuracy,precision_clase_1,recall_clase_1,f1_clase_1
0,KNN,0.712996,0.643243,0.561321,0.599496
1,Regresión Logística,0.711191,0.635417,0.575472,0.603960
2,Árbol de Decisión,0.678700,0.578704,0.589623,0.584112
3,Random Forest,0.711191,0.634021,0.580189,0.605911
4,Gradient Boosting,0.691336,0.601990,0.570755,0.585956


In [12]:
# Ordenamos por recall_clase_1:
resultados_modelos.sort_values(by="recall_clase_1", ascending=False)


,modelo,accuracy,precision_clase_1,recall_clase_1,f1_clase_1
2,Árbol de Decisión,0.678700,0.578704,0.589623,0.584112
3,Random Forest,0.711191,0.634021,0.580189,0.605911
1,Regresión Logística,0.711191,0.635417,0.575472,0.603960
4,Gradient Boosting,0.691336,0.601990,0.570755,0.585956
0,KNN,0.712996,0.643243,0.561321,0.599496


In [13]:
# Ordenamos por f1_clase_1:
resultados_modelos.sort_values(by="f1_clase_1", ascending=False)

,modelo,accuracy,precision_clase_1,recall_clase_1,f1_clase_1
3,Random Forest,0.711191,0.634021,0.580189,0.605911
1,Regresión Logística,0.711191,0.635417,0.575472,0.603960
0,KNN,0.712996,0.643243,0.561321,0.599496
4,Gradient Boosting,0.691336,0.601990,0.570755,0.585956
2,Árbol de Decisión,0.678700,0.578704,0.589623,0.584112


## Criterio de selección del modelo

En este proyecto, el objetivo principal es anticipar tickets con riesgo de incumplir SLA.

Por eso, la métrica más importante es el `recall` de la clase `1`.

Un recall alto en la clase `1` significa que el modelo detecta una mayor proporción de tickets que realmente incumplen SLA.

Desde el punto de vista de negocio, el falso negativo es el error más costoso, porque representa un ticket que realmente incumplió SLA pero que el modelo no detectó a tiempo.

Sin embargo, también se revisará el F1-score para asegurar que el modelo mantenga un equilibrio razonable entre detección de riesgos y falsas alarmas.

In [14]:
# Identificamos el mejor modelo según recall:
mejor_modelo_recall = resultados_modelos.sort_values(
    by="recall_clase_1",
    ascending=False
).iloc[0]

mejor_modelo_recall

modelo               Árbol de Decisión
accuracy                        0.6787
precision_clase_1             0.578704
recall_clase_1                0.589623
f1_clase_1                    0.584112
Name: 2, dtype: object

In [15]:
# Identificamos el mejor modelo según F1-score:
mejor_modelo_f1 = resultados_modelos.sort_values(
    by="f1_clase_1",
    ascending=False
).iloc[0]

mejor_modelo_f1

modelo               Random Forest
accuracy                  0.711191
precision_clase_1         0.634021
recall_clase_1            0.580189
f1_clase_1                0.605911
Name: 3, dtype: object

## Interpretación de la comparación de modelos

La comparación inicial muestra que los modelos tienen rendimientos relativamente cercanos, pero con diferencias importantes según la métrica analizada.

Aunque la accuracy permite medir el rendimiento general, en este proyecto se prioriza el recall de la clase `1`, porque la clase `1` representa los tickets que incumplieron SLA.

El modelo con mayor recall para la clase `1` fue el Árbol de Decisión, con un recall de 0,5896. Esto significa que fue el modelo que detectó una mayor proporción de tickets que realmente incumplieron SLA.

Sin embargo, el Árbol de Decisión también obtuvo una accuracy menor que otros modelos, lo que indica un rendimiento global menos estable.

Por otro lado, Random Forest obtuvo el mejor F1-score para la clase `1`, con un valor de 0,6059. Esto sugiere que ofrece el mejor equilibrio entre precision y recall para detectar incumplimientos de SLA.

Desde una perspectiva de negocio, no basta con maximizar el recall si el modelo genera demasiadas falsas alarmas. El modelo debe detectar casos críticos, pero también mantener un nivel razonable de precisión para que el equipo operativo pueda confiar en las alertas.

Por este motivo, Random Forest se considera el modelo candidato más sólido para una siguiente fase de optimización.

## Conclusión del notebook

En este notebook se entrenaron y compararon cinco modelos de clasificación supervisada:

- KNN.
- Regresión Logística.
- Árbol de Decisión.
- Random Forest.
- Gradient Boosting.

La comparación se realizó utilizando métricas relevantes para el problema de negocio: accuracy, precision de clase `1`, recall de clase `1` y F1-score de clase `1`.

El Árbol de Decisión obtuvo el mejor recall para la clase `1`, lo que indica una mayor capacidad para detectar tickets que incumplen SLA.

Sin embargo, Random Forest obtuvo el mejor F1-score para la clase `1`, mostrando un mejor equilibrio entre detección de incumplimientos y control de falsas alarmas.

Por tanto, Random Forest se selecciona como modelo candidato para la siguiente fase de optimización y ajuste de hiperparámetros.